# Frost Training for Xent Games

In this notebook, we will train a model on a Xent Game using the efficient [Frost algorithm](https://arxiv.org/abs/2605.27701v1). The game we are considering is the following: Given a text (a short story, an article, ...), find a title that minimizes the cross-entropy of that text under a judge LLM. Concretely, let $s$ be the first 128 tokens of a text and let $t$ be a 10-token string produced by the player. The reward is given by
$$\log \mathbb{P}(s \mid \text{"Title:"} + t + \text{"\textbackslash n\textbackslash n"}),$$
where $\mathbb{P}$ is the likelihood function of the judge LLM. $t$ may not contains any tokens from $s$.

We use texts from [Cosmopedia](https://huggingface.co/datasets/HuggingFaceTB/cosmopedia) (synthetic stories, textbooks, etc.) and [FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) (educational web pages).

## Training

We use a Qwen3.5 model (2B, 4B, 9B) as the player that we train, and the same, but frozen model as the judge.
- We train a LoRA adapter with rank 64.
- We use 4 texts per training step with 4 rollouts each, enhanced with 4 Frost candidates.
- The first 16 texts from the dataset serve as validation data.

The 2B model can be trained with 15 GB of VRAM, e.g., on the free Colab T4 GPU; the others need more VRAM.

Training the 2B model with a micro batch size of 4 takes ~30s for 10 steps on an A100 with 40 GB of VRAM. Running the model for the first time will take a little longer due to the initial setup.

In [ ]:
from html import escape

import matplotlib.pyplot as plt
import ipywidgets as ipw
from IPython.display import HTML, clear_output, display

from widgets import dataset_dropdown, micro_batch_size_dropdown, model_dropdown, steps_dropdown
from training import train_with_frost

In [ ]:
dataset_dropdown()

In [ ]:
model_dropdown()

In [ ]:
steps_dropdown()

In [ ]:
# Higher is faster, but requires more memory
# Keep at 1 for Colab
micro_batch_size_dropdown()

In [ ]:
# True is faster, but requires more memory
batch_taylor_approximations = True

results = train_with_frost(
    dataset=dataset,
    text_len=128,
    title_len=10,
    player_model=model,
    judge_model=model,
    num_steps=steps,
    num_texts_per_step=4,
    num_rollouts_per_text=4,
    num_frost_samples_per_text=4,
    micro_batch_size=micro_batch_size,
    lr=1e-5,
    kl_regularizer=0.1,
    probability_gate=1e-4,
    lora_rank=64,
    num_validation_texts=16,
    num_validation_rollouts=8,
    validation_steps=10,
    batch_taylor_approximations=batch_taylor_approximations,
)

trained_model = results.model
tokenizer = results.tokenizer

In [ ]:
mean_scores = []
best_scores = []
for step in results.validation_steps:
    scores = results.validation_scores[step]
    flat_scores = [score for row in scores for score in row]
    mean_scores.append(sum(flat_scores) / len(flat_scores))
    best_scores.append(sum(max(row) for row in scores) / len(scores))

plt.figure(figsize=(7, 4))
plt.plot(results.validation_steps, mean_scores, marker="o", label="Mean validation score")
plt.plot(
    results.validation_steps,
    best_scores,
    marker="o",
    label=f"Best of {results.config['num_validation_rollouts']}",
)
plt.xlabel("Training step")
plt.ylabel("Validation score")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
def _short_text(text, max_chars=90):
    one_line = " ".join(text.split())
    return one_line if len(one_line) <= max_chars else one_line[:max_chars - 1] + "..."


text_dropdown = ipw.Dropdown(
    options=[(_short_text(text), i) for i, text in enumerate(results.validation_texts)],
    description="Validation text",
    style={"description_width": "initial"},
    layout=ipw.Layout(width="max-content"),
)
table_output = ipw.Output()


def _sorted_rows(step, text_index):
    outputs = results.validation_outputs[step][text_index]
    scores = results.validation_scores[step][text_index]
    rows = list(zip(outputs, scores))
    return sorted(rows, key=lambda row: row[1], reverse=True)


def _table_html(title, rows):
    body = "".join(
        "<tr>"
        f"<td style='white-space: pre-wrap; word-break: break-word'>{escape(output)}</td>"
        f"<td>{score:.2f}</td>"
        "</tr>"
        for output, score in rows
    )
    return (
        f"<h4>{escape(title)}</h4>"
        "<table>"
        "<thead><tr><th>Title</th><th>Score</th></tr></thead>"
        f"<tbody>{body}</tbody>"
        "</table>"
    )


def show_validation_outputs(change=None):
    text_index = text_dropdown.value
    first_step = results.validation_steps[0]
    final_step = results.validation_steps[-1]

    with table_output:
        clear_output(wait=True)
        text = escape(results.validation_texts[text_index])
        display(HTML(f"<h4>Validation text {text_index}</h4><pre style='white-space: pre-wrap'>{text}</pre>"))
        display(HTML(_table_html("Before training", _sorted_rows(first_step, text_index))))
        display(HTML(_table_html("After training", _sorted_rows(final_step, text_index))))


text_dropdown.observe(show_validation_outputs, names="value")
display(text_dropdown, table_output)
show_validation_outputs()